# S01 · RadixAttention vs hash-chained prefix caching

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 8. Sources: SGLang (arXiv:2312.07104, github.com/sgl-project/sglang); vLLM automatic prefix caching.

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Rebuild both prefix-cache designs and compare hit rates and memory on shared-prompt and
multi-turn workloads. Official code: `sglang/srt/mem_cache/radix_cache.py`,
`vllm/v1/core/kv_cache_manager.py`.

## Theory and equations
Hash chain: $h_i = H(h_{i-1}, x_{iB..(i+1)B})$, so only full blocks with identical prefixes match.
Radix tree: longest-prefix match at token granularity; lock counts; LRU leaf eviction.

## Diagram
```
root --[system prompt 2000, lock 3]--+--[A: q1]--[A: r1+q2]
                                     +--[B: q]
                                     +--[C: q, lock 0 -> evictable]
```

## Implementation

In [ ]:
import random
from inference_lab.prefix import HashBlockCache, RadixCache
rng = random.Random(0)
system = list(range(10_000, 12_000))
users = [system + [rng.randint(0, 9999) for _ in range(200)] for _ in range(50)]

## Experiment 1: shared system prompt (*measured*, CPU)

In [ ]:
hc = HashBlockCache(20_000, 16); rc = RadixCache(10**7)
h_hits = sum(hc.acquire(u)[1] for u in users)
r_hits = sum(rc.insert(u)[0] for u in users)
total = sum(map(len, users))
print(f"prompt tokens {total:,}; hash hits {h_hits:,} ({h_hits/total:.1%}); radix hits {r_hits:,} ({r_hits/total:.1%})")
print(f"hash blocks used {hc.capacity - len(hc.free)} vs {sum(-(-len(u)//16) for u in users)} without sharing")
print(f"radix cached tokens {rc.cached_tokens():,} vs {total:,} without sharing")

## Experiment 2: multi-turn chat and eviction pressure (*measured*)

In [ ]:
def multiturn(capacity_tokens):
    rng = random.Random(1); rc = RadixCache(capacity_tokens); hits = tot = 0
    convs = [list(system[:500]) for _ in range(20)]
    for turn in range(6):
        for c in convs:
            c += [rng.randint(0, 9999) for _ in range(150)]
            h, path = rc.insert(list(c)); hits += h; tot += len(c); rc.release(path)
            c += [rng.randint(0, 9999) for _ in range(250)]
    return hits / tot, rc.evicted_tokens
caps = [5_000, 10_000, 20_000, 40_000, 80_000]
res = [multiturn(c) for c in caps]
for c, (hr, ev) in zip(caps, res): print(f"capacity {c:6,} tokens: hit rate {hr:.1%}, evicted {ev:,}")
plt.plot(caps, [100 * r[0] for r in res], "o-", color=C["shared"]); plt.xscale("log")
plt.xlabel("cache capacity (tokens)"); plt.ylabel("prompt tokens served from cache (%)"); plt.show()

## Results and interpretation
Both designs reuse the shared prompt fully. The radix tree also reuses partial blocks.
Hit rate collapses once the hot working set exceeds capacity: LRU starts evicting
conversations that are about to send their next turn.

## Limitations
Generated replies are not inserted into the cache here (SGLang does insert them), so
multi-turn hit rates are a lower bound. No GPU memory or kernels.

## Conclusion
Prefix caching is a data-structure problem; the policy question is eviction under pressure.